# 02 — Data Cleaning & SQLite Loading

Clean all 10 datasets, design star schema, load into SQLite, verify row counts.

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import os

os.chdir('/Users/maneeshreddy/Desktop/mutual_fund_analysis')
RAW_DIR = 'data_1/Bluestock_MF_Datasets'
PROCESSED_DIR = 'data/processed'
DB_PATH = 'data/db/bluestock_mf.db'
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs('data/db', exist_ok=True)
print('Directories ready.')

## Clean NAV History

In [ ]:
df = pd.read_csv(f'{RAW_DIR}/02_nav_history.csv')
print('Raw:', len(df), 'rows')

df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['amfi_code'] = pd.to_numeric(df['amfi_code'], errors='coerce').astype(int)
df['nav'] = pd.to_numeric(df['nav'], errors='coerce')
df = df.dropna(subset=['date', 'amfi_code', 'nav'])
df = df[df['nav'] > 0]
df = df.drop_duplicates(subset=['amfi_code', 'date'], keep='first')
df = df.sort_values(['amfi_code', 'date']).reset_index(drop=True)

# Forward-fill holidays/weekends
filled = []
for code in df['amfi_code'].unique():
    cdf = df[df['amfi_code'] == code].copy()
    bdays = pd.bdate_range(cdf['date'].min(), cdf['date'].max())
    full = pd.DataFrame({'date': bdays, 'amfi_code': code})
    merged = full.merge(cdf, on=['amfi_code', 'date'], how='left')
    merged['nav'] = merged['nav'].ffill()
    filled.append(merged.dropna(subset=['nav']))
df_filled = pd.concat(filled, ignore_index=True)

df_filled.to_csv(f'{PROCESSED_DIR}/nav_history_cleaned.csv', index=False)
print('Cleaned NAV:', len(df_filled), 'rows')

## Clean All Other Datasets

In [ ]:
# Fund Master
fm = pd.read_csv(f'{RAW_DIR}/01_fund_master.csv')
fm['amfi_code'] = pd.to_numeric(fm['amfi_code'], errors='coerce').astype(int)
fm['launch_date'] = pd.to_datetime(fm['launch_date'], errors='coerce')
fm['expense_ratio_pct'] = pd.to_numeric(fm['expense_ratio_pct'], errors='coerce')
fm.to_csv(f'{PROCESSED_DIR}/fund_master_cleaned.csv', index=False)
print('Fund master:', len(fm), 'rows')

# AUM by Fund House
aum = pd.read_csv(f'{RAW_DIR}/03_aum_by_fund_house.csv')
aum['date'] = pd.to_datetime(aum['date'], errors='coerce')
aum['aum_lakh_crore'] = pd.to_numeric(aum['aum_lakh_crore'], errors='coerce')
aum['aum_crore'] = pd.to_numeric(aum['aum_crore'], errors='coerce')
aum.to_csv(f'{PROCESSED_DIR}/aum_by_fund_house_cleaned.csv', index=False)
print('AUM:', len(aum), 'rows')

# SIP Inflows
sip = pd.read_csv(f'{RAW_DIR}/04_monthly_sip_inflows.csv')
sip['month'] = pd.to_datetime(sip['month'], errors='coerce')
for c in ['sip_inflow_crore', 'active_sip_accounts_crore', 'new_sip_accounts_lakh', 'sip_aum_lakh_crore', 'yoy_growth_pct']:
    sip[c] = pd.to_numeric(sip[c], errors='coerce')
sip.to_csv(f'{PROCESSED_DIR}/monthly_sip_inflows_cleaned.csv', index=False)
print('SIP:', len(sip), 'rows')

# Category Inflows
cat = pd.read_csv(f'{RAW_DIR}/05_category_inflows.csv')
cat['month'] = pd.to_datetime(cat['month'], errors='coerce')
cat['net_inflow_crore'] = pd.to_numeric(cat['net_inflow_crore'], errors='coerce')
cat.to_csv(f'{PROCESSED_DIR}/category_inflows_cleaned.csv', index=False)
print('Category:', len(cat), 'rows')

# Folio Count
folio = pd.read_csv(f'{RAW_DIR}/06_industry_folio_count.csv')
folio['month'] = pd.to_datetime(folio['month'], errors='coerce')
for c in ['total_folios_crore', 'equity_folios_crore', 'debt_folios_crore', 'hybrid_folios_crore', 'others_folios_crore']:
    folio[c] = pd.to_numeric(folio[c], errors='coerce')
folio.to_csv(f'{PROCESSED_DIR}/industry_folio_count_cleaned.csv', index=False)
print('Folio:', len(folio), 'rows')

# Scheme Performance
perf = pd.read_csv(f'{RAW_DIR}/07_scheme_performance.csv')
perf['amfi_code'] = pd.to_numeric(perf['amfi_code'], errors='coerce').astype(int)
for c in ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating']:
    perf[c] = pd.to_numeric(perf[c], errors='coerce')
perf.to_csv(f'{PROCESSED_DIR}/scheme_performance_cleaned.csv', index=False)
print('Performance:', len(perf), 'rows')

# Investor Transactions
txn = pd.read_csv(f'{RAW_DIR}/08_investor_transactions.csv')
txn['transaction_date'] = pd.to_datetime(txn['transaction_date'], errors='coerce')
txn['amfi_code'] = pd.to_numeric(txn['amfi_code'], errors='coerce').astype(int)
txn['amount_inr'] = pd.to_numeric(txn['amount_inr'], errors='coerce')
txn = txn[txn['amount_inr'] > 0]
txn.to_csv(f'{PROCESSED_DIR}/investor_transactions_cleaned.csv', index=False)
print('Transactions:', len(txn), 'rows')

# Portfolio Holdings
hold = pd.read_csv(f'{RAW_DIR}/09_portfolio_holdings.csv')
hold['amfi_code'] = pd.to_numeric(hold['amfi_code'], errors='coerce').astype(int)
hold['portfolio_date'] = pd.to_datetime(hold['portfolio_date'], errors='coerce')
for c in ['weight_pct', 'market_value_cr', 'current_price_inr']:
    hold[c] = pd.to_numeric(hold[c], errors='coerce')
hold.to_csv(f'{PROCESSED_DIR}/portfolio_holdings_cleaned.csv', index=False)
print('Holdings:', len(hold), 'rows')

# Benchmark Indices
bench = pd.read_csv(f'{RAW_DIR}/10_benchmark_indices.csv')
bench['date'] = pd.to_datetime(bench['date'], errors='coerce')
bench['close_value'] = pd.to_numeric(bench['close_value'], errors='coerce')
bench.to_csv(f'{PROCESSED_DIR}/benchmark_indices_cleaned.csv', index=False)
print('Benchmark:', len(bench), 'rows')

print('\nAll datasets cleaned and saved.')

## Create SQLite Database

In [ ]:
with open('schema.sql', 'r') as f:
    schema_sql = f.read()

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
engine = create_engine(f'sqlite:///{DB_PATH}')

with engine.connect() as conn:
    for statement in schema_sql.split(';'):
        stmt = statement.strip()
        if stmt and not stmt.startswith('--'):
            try:
                conn.execute(text(stmt))
            except Exception:
                pass
    conn.commit()
print('Schema created.')

In [ ]:
# Populate dim_date
all_dates = set()
for col, f in [('date', 'nav_history_cleaned.csv'), ('transaction_date', 'investor_transactions_cleaned.csv'),
               ('date', 'aum_by_fund_house_cleaned.csv'), ('date', 'benchmark_indices_cleaned.csv')]:
    df = pd.read_csv(f'{PROCESSED_DIR}/{f}', usecols=[col])
    all_dates.update(pd.to_datetime(df[col]).tolist())

min_d, max_d = min(all_dates), max(all_dates)
dates = pd.date_range(min_d, max_d, freq='D')
dim_date = pd.DataFrame({'full_date': dates})
dim_date['date_key'] = dim_date['full_date'].dt.strftime('%Y%m%d').astype(int)
dim_date['year'] = dim_date['full_date'].dt.year
dim_date['quarter'] = dim_date['full_date'].dt.quarter
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['month_name'] = dim_date['full_date'].dt.month_name()
dim_date['day'] = dim_date['full_date'].dt.day
dim_date['day_of_week'] = dim_date['full_date'].dt.dayofweek
dim_date['day_name'] = dim_date['full_date'].dt.day_name()
dim_date['is_weekend'] = dim_date['day_of_week'].isin([5, 6])
dim_date['is_month_end'] = dim_date['full_date'].dt.is_month_end
dim_date['fiscal_year'] = dim_date['full_date'].apply(lambda x: x.year if x.month >= 4 else x.year - 1)
dim_date.to_sql('dim_date', engine, if_exists='replace', index=False)
print('dim_date:', len(dim_date), 'rows')

In [ ]:
# Load all fact tables
tables = [
    ('fund_master_cleaned.csv', 'dim_fund', None),
    ('nav_history_cleaned.csv', 'fact_nav', ['amfi_code', 'date', 'nav']),
    ('investor_transactions_cleaned.csv', 'fact_transactions', None),
    ('scheme_performance_cleaned.csv', 'fact_performance', None),
    ('aum_by_fund_house_cleaned.csv', 'fact_aum', None),
    ('monthly_sip_inflows_cleaned.csv', 'fact_sip_inflows', None),
    ('category_inflows_cleaned.csv', 'fact_category_inflows', None),
    ('industry_folio_count_cleaned.csv', 'fact_folio_count', None),
    ('portfolio_holdings_cleaned.csv', 'fact_portfolio_holdings', None),
    ('benchmark_indices_cleaned.csv', 'fact_benchmark_indices', None),
]

for csv, table, cols in tables:
    df = pd.read_csv(f'{PROCESSED_DIR}/{csv}', usecols=cols) if cols else pd.read_csv(f'{PROCESSED_DIR}/{csv}')
    df.to_sql(table, engine, if_exists='replace', index=False)
    print(table + ':', len(df), 'rows')

engine.dispose()
print('\nDatabase loaded successfully.')

In [ ]:
# Verify row counts
engine = create_engine(f'sqlite:///{DB_PATH}')
for table in ['dim_fund', 'dim_date', 'fact_nav', 'fact_transactions', 'fact_performance',
              'fact_aum', 'fact_sip_inflows', 'fact_category_inflows', 'fact_folio_count',
              'fact_portfolio_holdings', 'fact_benchmark_indices']:
    cnt = pd.read_sql('SELECT COUNT(*) as c FROM ' + table, engine).iloc[0]['c']
    print(table + ':', cnt, 'rows')
engine.dispose()